In [0]:
import time

dbutils.widgets.text("catalog", "databricks_dev")
CATALOG = dbutils.widgets.get("catalog")
spark.sql(f"USE CATALOG {CATALOG}")

print(f"Catalog: {CATALOG}")
print(f"Tables we'll optimize:")
print(f"  - {CATALOG}.gold.fact_sales")
print(f"  - {CATALOG}.silver.silver_orders")
print(f"  - {CATALOG}.silver.silver_customers")

In [0]:
%sql
OPTIMIZE databricks_dev.gold.fact_sales
ZORDER BY (customer_id, product_id);

In [0]:
%sql
-- Optimize silver orders by customer_id (common join key)
OPTIMIZE databricks_dev.silver.silver_orders
ZORDER BY (customer_id);

In [0]:
%sql
-- Optimize silver order_items by order_id and product_id
OPTIMIZE databricks_dev.silver.silver_order_items
ZORDER BY (order_id, product_id);

In [0]:
%sql
VACUUM databricks_dev.gold.fact_sales RETAIN 168 HOURS;
VACUUM databricks_dev.silver.silver_orders RETAIN 168 HOURS;
VACUUM databricks_dev.silver.silver_order_items RETAIN 168 HOURS;
VACUUM databricks_dev.silver.silver_customers RETAIN 168 HOURS;

In [0]:
start = time.time()
result = spark.sql(f"""
    SELECT customer_id, COUNT(*) as orders, SUM(line_total) as revenue
    FROM {CATALOG}.gold.fact_sales
    WHERE customer_id = 1001
    GROUP BY customer_id
""").collect()
elapsed = time.time() - start

print(f"Query on Z-ORDERed column (customer_id = 1001): {elapsed:.2f}s")
print(f"Z-ORDER means Spark skips files that don't contain customer 1001")
print(f"Without Z-ORDER, Spark would scan ALL files")